# Comparative Analysis of GAN Loss Functions
## Complete Visualization Notebook

**Author:** Sanskar Kushwah | NIT Srinagar  
**Datasets:** CIFAR-10 · EuroSAT · CheXpert  
**Models:** Standard GAN · LSGAN · WGAN · WGAN-GP · Hinge Loss · Hybrid (Ours)

---
### Notebook Structure
| Cell | Plot | Description |
|------|------|-------------|
| 2 | Setup | Install packages, imports, palette, helpers |
| 3 | Load Data | Read all 4 CSV files |
| 4 | Plot 01 | FID curves over epochs — CIFAR-10 |
| 5 | Plot 02 | FID curves over epochs — EuroSAT |
| 6 | Plot 03 | G & D Loss curves (6 subplots) — CIFAR-10 |
| 7 | Plot 04 | G & D Loss curves (6 subplots) — EuroSAT |
| 8 | Plot 05 | Mode Variance over epochs — CIFAR-10 |
| 9 | Plot 06 | Mode Variance over epochs — EuroSAT |
| 10 | Plot 07 | Best FID bar chart — all 3 datasets |
| 11 | Plot 08 | Mode Variance bar — CIFAR-10 vs EuroSAT |
| 12 | Plot 09 | Quality–Diversity scatter (FID vs ModeVar) |
| 13 | Plot 10 | CheXpert Hybrid — Loss + FID + ModeVar |
| 14 | Save All | Run all plots and save to ./plots/ |

> **CSV files required** (put them in a `csv/` folder next to this notebook):
> - `csv/cifer_all_6.csv`
> - `csv/euro_5229_graph.csv`
> - `csv/euro_hybrid.csv`
> - `csv/hybridloss_cxpert.csv`


In [16]:
# ── Install dependencies (run once) ─────────────────────────────────────────
# import subprocess, sys
# pkgs = ["numpy", "pandas", "matplotlib", "scipy"]
# for p in pkgs:
#     subprocess.check_call([sys.executable, "-m", "pip", "install", p, "-q"])
# print("All packages ready.")


## Cell 2 — Setup: imports, palette, helpers

In [17]:
import os, warnings
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.lines import Line2D
from scipy.ndimage import uniform_filter1d

warnings.filterwarnings("ignore")
os.makedirs("plots", exist_ok=True)

# ── Global plot style ─────────────────────────────────────────────────────────
matplotlib.rcParams.update({
    "font.family":        "DejaVu Sans",
    "axes.spines.top":    False,
    "axes.spines.right":  False,
    "axes.grid":          True,
    "grid.alpha":         0.22,
    "grid.linestyle":     "--",
    "grid.linewidth":     0.6,
    "figure.dpi":         120,
    "savefig.dpi":        300,
    "savefig.bbox":       "tight",
    "savefig.pad_inches": 0.15,
    "axes.titlesize":     13,
    "axes.labelsize":     11,
    "xtick.labelsize":    9,
    "ytick.labelsize":    9,
})

# ── Color palette — one color per model ──────────────────────────────────────
PAL = {
    "STANDARD": "#E24B4A",   # red
    "LSGAN":    "#EF9F27",   # amber
    "WGAN":     "#378ADD",   # blue
    "WGANGP":   "#1D9E75",   # teal/green
    "HINGE":    "#7F77DD",   # purple
    "HYBRID":   "#D85A30",   # coral  ← proposed method
}
LABELS = {
    "STANDARD": "Standard GAN",
    "LSGAN":    "LSGAN",
    "WGAN":     "WGAN",
    "WGANGP":   "WGAN-GP",
    "HINGE":    "Hinge Loss",
    "HYBRID":   "Hybrid (Ours)",
}
ORDER = list(PAL.keys())

def col(key):  return PAL[key]
def lbl(key):  return LABELS[key]

def smooth(arr, w=5):
    """Simple moving-average smoother for noisy loss curves."""
    arr = np.array(arr, dtype=float)
    return uniform_filter1d(arr, size=min(w, len(arr)))

def save_fig(fig, name):
    path = f"plots/{name}.png"
    fig.savefig(path)
    print(f"  saved → {path}")

print("Setup complete. Palette and helpers loaded.")


Setup complete. Palette and helpers loaded.


## Cell 3 — Load CSV data

In [18]:
# ── Load all four CSV files ───────────────────────────────────────────────────
CIFAR    = pd.read_csv("csv/cifer_all_6.csv")
EURO150  = pd.read_csv("csv/euro_5229_graph.csv")
EURO_HYB = pd.read_csv("csv/euro_hybrid.csv")
CHEX_HYB = pd.read_csv("csv/hybridloss_cxpert.csv")

# ── Helper: extract a column for one experiment ───────────────────────────────
def get_col(df, exp, col_name):
    if "dataset" in df.columns:          # euro_hybrid has no Experiment column
        return df[col_name].tolist()
    return df[df["Experiment"] == exp][col_name].tolist()

# ── Helper: extract FID checkpoints (epoch, fid) ─────────────────────────────
def get_fid(df, exp):
    if "dataset" in df.columns:
        sub = df[df["FID"].notna()]
    else:
        sub = df[(df["Experiment"] == exp) & df["FID"].notna()]
    return sub["Epoch"].tolist(), sub["FID"].tolist()
# ── Quick sanity check ────────────────────────────────────────────────────────
print("CIFAR-10   experiments:", CIFAR["Experiment"].unique().tolist())
print("EuroSAT    experiments:", EURO150["Experiment"].unique().tolist())
print("Euro Hybrid rows:", len(EURO_HYB))
print("CheXpert   rows:", len(CHEX_HYB))


CIFAR-10   experiments: ['STANDARD', 'LSGAN', 'WGAN', 'WGANGP', 'HINGE', 'HYBRID']
EuroSAT    experiments: ['STANDARD', 'LSGAN', 'WGAN', 'WGANGP', 'HINGE']
Euro Hybrid rows: 150
CheXpert   rows: 100


## Plot 01 — FID curves over epochs · CIFAR-10

In [19]:
fig, ax = plt.subplots(figsize=(11, 5.5))

for key in ORDER:
    ep, fid = get_fid(CIFAR, key)
    lw = 2.5 if key == "HYBRID" else 1.8
    ls = "--" if key == "HYBRID" else "-"
    ms = 6   if key == "HYBRID" else 4
    ax.plot(ep, fid, color=col(key), lw=lw, ls=ls,
            marker="o", markersize=ms, label=lbl(key))

ax.set_xlabel("Epoch")
ax.set_ylabel("FID score  (↓ lower is better)")
ax.set_title("FID Score over Training Epochs — CIFAR-10  (real data)")
ax.legend(fontsize=9, ncol=2, framealpha=0.85)
plt.tight_layout()
plt.show()
save_fig(fig, "01_fid_curves_cifar10")


  saved → plots/01_fid_curves_cifar10.png


## Plot 02 — FID curves over epochs · EuroSAT

In [20]:
fig, ax = plt.subplots(figsize=(11, 5.5))

for key in ["STANDARD", "LSGAN", "WGAN", "WGANGP", "HINGE"]:
    ep, fid = get_fid(EURO150, key)
    ax.plot(ep, fid, color=col(key), lw=1.8, marker="o",
            markersize=4, label=lbl(key))

# Hybrid comes from separate file
fid_eh = EURO_HYB[EURO_HYB["FID"].notna()]
ax.plot(fid_eh["Epoch"].tolist(), fid_eh["FID"].tolist(),
        color=col("HYBRID"), lw=2.5, ls="--", marker="o",
        markersize=6, label=lbl("HYBRID"))

ax.set_xlabel("Epoch")
ax.set_ylabel("FID score  (↓ lower is better)")
ax.set_title("FID Score over Training Epochs — EuroSAT  (real data)")
ax.legend(fontsize=9, ncol=2, framealpha=0.85)
plt.tight_layout()
plt.show()
save_fig(fig, "02_fid_curves_eurosat")


  saved → plots/02_fid_curves_eurosat.png


## Plot 03 — Generator & Discriminator Loss Curves · CIFAR-10

In [21]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for ax, key in zip(axes, ORDER):
    ep   = CIFAR[CIFAR["Experiment"] == key]["Epoch"].tolist()
    gl_s = smooth(get_col(CIFAR, key, "G_Loss"))
    dl_s = smooth(get_col(CIFAR, key, "D_Loss"))

    ax.plot(ep, gl_s, color=col(key), lw=2.0, label="Generator")
    ax.plot(ep, dl_s, color=col(key), lw=1.4, ls="--", alpha=0.6,
            label="Discriminator")
    ax.set_title(lbl(key), color=col(key), fontweight="bold")
    ax.set_xlabel("Epoch", fontsize=9)
    ax.set_ylabel("Loss",  fontsize=9)
    ax.legend(fontsize=8, framealpha=0.7)

fig.suptitle("Generator & Discriminator Loss Curves — CIFAR-10\n"
             "Solid = Generator    Dashed = Discriminator",
             fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()
save_fig(fig, "03_loss_curves_cifar10")


  saved → plots/03_loss_curves_cifar10.png


## Plot 04 — Generator & Discriminator Loss Curves · EuroSAT

In [22]:
dfs = {k: EURO150[EURO150["Experiment"] == k]
       for k in ["STANDARD","LSGAN","WGAN","WGANGP","HINGE"]}
dfs["HYBRID"] = EURO_HYB

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for ax, key in zip(axes, ORDER):
    sub  = dfs[key]
    ep   = sub["Epoch"].tolist()
    gl_s = smooth(sub["G_Loss"].tolist())
    dl_s = smooth(sub["D_Loss"].tolist())

    ax.plot(ep, gl_s, color=col(key), lw=2.0, label="Generator")
    ax.plot(ep, dl_s, color=col(key), lw=1.4, ls="--", alpha=0.6,
            label="Discriminator")
    ax.set_title(lbl(key), color=col(key), fontweight="bold")
    ax.set_xlabel("Epoch", fontsize=9)
    ax.set_ylabel("Loss",  fontsize=9)
    ax.legend(fontsize=8, framealpha=0.7)

fig.suptitle("Generator & Discriminator Loss Curves — EuroSAT\n"
             "Solid = Generator    Dashed = Discriminator",
             fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()
save_fig(fig, "04_loss_curves_eurosat")


  saved → plots/04_loss_curves_eurosat.png


## Plot 05 — Mode Variance over epochs · CIFAR-10

In [23]:
fig, ax = plt.subplots(figsize=(11, 5.5))

for key in ORDER:
    ep = CIFAR[CIFAR["Experiment"] == key]["Epoch"].tolist()
    mv = smooth(get_col(CIFAR, key, "ModeVar"), w=7)
    lw = 2.5 if key == "HYBRID" else 1.8
    ls = "--" if key == "HYBRID" else "-"
    ax.plot(ep, mv, color=col(key), lw=lw, ls=ls, label=lbl(key))

ax.set_xlabel("Epoch")
ax.set_ylabel("Mode Variance  (↑ higher = more diverse)")
ax.set_title("Mode Variance over Training — CIFAR-10  (real data)")
ax.legend(fontsize=9, ncol=2, framealpha=0.85)
plt.tight_layout()
plt.show()
save_fig(fig, "05_modevar_curves_cifar10")


  saved → plots/05_modevar_curves_cifar10.png


## Plot 06 — Mode Variance over epochs · EuroSAT

In [24]:
dfs = {k: EURO150[EURO150["Experiment"] == k]
       for k in ["STANDARD","LSGAN","WGAN","WGANGP","HINGE"]}
dfs["HYBRID"] = EURO_HYB

fig, ax = plt.subplots(figsize=(11, 5.5))

for key in ORDER:
    sub = dfs[key]
    ep  = sub["Epoch"].tolist()
    mv  = smooth(sub["ModeVar"].tolist(), w=7)
    lw  = 2.5 if key == "HYBRID" else 1.8
    ls  = "--" if key == "HYBRID" else "-"
    ax.plot(ep, mv, color=col(key), lw=lw, ls=ls, label=lbl(key))

ax.set_xlabel("Epoch")
ax.set_ylabel("Mode Variance  (↑ higher = more diverse)")
ax.set_title("Mode Variance over Training — EuroSAT  (real data)")
ax.legend(fontsize=9, ncol=2, framealpha=0.85)
plt.tight_layout()
plt.show()
save_fig(fig, "06_modevar_curves_eurosat")


  saved → plots/06_modevar_curves_eurosat.png


## Plot 07 — Best FID bar chart · all datasets

In [25]:
# Collect best FID per model per dataset
best_cifar = {k: min(get_fid(CIFAR, k)[1]) for k in ORDER}

best_euro = {}
for k in ["STANDARD","LSGAN","WGAN","WGANGP","HINGE"]:
    best_euro[k] = min(get_fid(EURO150, k)[1])
best_euro["HYBRID"] = float(EURO_HYB[EURO_HYB["FID"].notna()]["FID"].min())

best_chex = {k: None for k in ORDER}
best_chex["HYBRID"] = float(CHEX_HYB[CHEX_HYB["FID"].notna()]["FID"].min())

# Plot
x = np.arange(len(ORDER))
w = 0.26
fig, ax = plt.subplots(figsize=(13, 6))

b1 = ax.bar(x - w, [best_cifar[k] for k in ORDER], w,
            color=[col(k) for k in ORDER], edgecolor="white",
            linewidth=0.8, label="CIFAR-10", zorder=3)
b2 = ax.bar(x,     [best_euro[k]  for k in ORDER], w,
            color=[col(k) for k in ORDER], alpha=0.6,
            edgecolor="white", linewidth=0.8, hatch="///",
            label="EuroSAT", zorder=3)
b3 = ax.bar(x + w, [best_chex[k] if best_chex[k] else 0 for k in ORDER], w,
            color=[col(k) for k in ORDER], alpha=0.35,
            edgecolor="white", linewidth=0.8, hatch="xxx",
            label="CheXpert (Hybrid only)", zorder=3)

for bar in list(b1) + list(b2) + list(b3):
    v = bar.get_height()
    if v > 1:
        ax.text(bar.get_x() + bar.get_width()/2, v + 1.5,
                f"{v:.1f}", ha="center", va="bottom",
                fontsize=7.5, color="#444")

ax.set_xticks(x)
ax.set_xticklabels([lbl(k) for k in ORDER], rotation=18, ha="right")
ax.set_ylabel("Best FID  (↓ lower is better)")
ax.set_title("Best FID Score by Loss Function — All Datasets  (real data)")

handles = [
    Line2D([0],[0], color="#666", lw=8, alpha=1.0,  label="CIFAR-10"),
    Line2D([0],[0], color="#666", lw=8, alpha=0.6,  label="EuroSAT"),
    Line2D([0],[0], color="#666", lw=8, alpha=0.35, label="CheXpert (Hybrid only)"),
]
ax.legend(handles=handles, fontsize=9, framealpha=0.9)
plt.tight_layout()
plt.show()
save_fig(fig, "07_best_fid_all_datasets")


  saved → plots/07_best_fid_all_datasets.png


## Plot 08 — Mode Variance bar · CIFAR-10 vs EuroSAT

In [26]:
# Final-epoch mode variance per model
mv_cifar = {k: CIFAR[CIFAR["Experiment"]==k]["ModeVar"].iloc[-1] for k in ORDER}
mv_euro  = {k: EURO150[EURO150["Experiment"]==k]["ModeVar"].iloc[-1]
             for k in ["STANDARD","LSGAN","WGAN","WGANGP","HINGE"]}
mv_euro["HYBRID"] = EURO_HYB["ModeVar"].iloc[-1]

x = np.arange(len(ORDER))
w = 0.35
fig, ax = plt.subplots(figsize=(12, 5.5))

b1 = ax.bar(x - w/2, [mv_cifar[k] for k in ORDER], w,
            color=[col(k) for k in ORDER], edgecolor="white",
            linewidth=0.8, label="CIFAR-10", zorder=3)
b2 = ax.bar(x + w/2, [mv_euro[k]  for k in ORDER], w,
            color=[col(k) for k in ORDER], alpha=0.55,
            edgecolor="white", linewidth=0.8, hatch="///",
            label="EuroSAT", zorder=3)

for bar in list(b1) + list(b2):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.003,
            f"{bar.get_height():.3f}",
            ha="center", va="bottom", fontsize=7.5, color="#444")

ax.set_xticks(x)
ax.set_xticklabels([lbl(k) for k in ORDER], rotation=18, ha="right")
ax.set_ylabel("Mode Variance  (↑ higher = more diverse)")
ax.set_title("Final Mode Variance — CIFAR-10 vs EuroSAT  (real data)")

handles = [
    Line2D([0],[0], color="#666", lw=8, alpha=1.0,  label="CIFAR-10"),
    Line2D([0],[0], color="#666", lw=8, alpha=0.55, label="EuroSAT"),
]
ax.legend(handles=handles, fontsize=9, framealpha=0.9)
plt.tight_layout()
plt.show()
save_fig(fig, "08_modevar_bar_comparison")


  saved → plots/08_modevar_bar_comparison.png


## Plot 09 — Quality–Diversity Scatter · CIFAR-10

In [27]:
best_fid = {k: min(get_fid(CIFAR, k)[1]) for k in ORDER}
final_mv = {k: CIFAR[CIFAR["Experiment"]==k]["ModeVar"].iloc[-1] for k in ORDER}

fig, ax = plt.subplots(figsize=(9, 6))

for k in ORDER:
    sz  = 280 if k == "HYBRID" else 120
    mrk = "*"  if k == "HYBRID" else "o"
    ax.scatter(best_fid[k], final_mv[k],
               color=col(k), s=sz, marker=mrk,
               edgecolors="white", linewidths=0.8, zorder=5)
    dy = -14 if k == "STANDARD" else 5
    ax.annotate(lbl(k), (best_fid[k], final_mv[k]),
                textcoords="offset points", xytext=(5, dy),
                fontsize=9, color=col(k), fontweight="bold")

ax.set_xlabel("Best FID  (← lower is better)")
ax.set_ylabel("Final Mode Variance  (↑ higher = more diverse)")
ax.set_title("Quality–Diversity Trade-off — CIFAR-10  (real data)\n"
             "Ideal: bottom-right corner  ★ = Hybrid (Ours)")
plt.tight_layout()
plt.show()
save_fig(fig, "09_pareto_scatter_cifar10")


  saved → plots/09_pareto_scatter_cifar10.png


## Plot 10 — CheXpert Hybrid: Loss + FID + Mode Variance

In [28]:
ep       = CHEX_HYB["Epoch"].tolist()
gl_s     = smooth(CHEX_HYB["G_Loss"].tolist(), w=9)
dl_s     = smooth(CHEX_HYB["D_Loss"].tolist(), w=9)
mv_s     = smooth(CHEX_HYB["ModeVar"].tolist(), w=7)
fid_rows = CHEX_HYB[CHEX_HYB["FID"].notna()]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))




# Panel A — Loss curves
axes[0].plot(ep, gl_s, color=col("HYBRID"), lw=2.0, label="Generator")
axes[0].plot(ep, dl_s, color=col("HYBRID"), lw=1.5, ls="--",
             alpha=0.6, label="Discriminator")
axes[0].set_title("A  Loss curves", fontweight="bold")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend(fontsize=9)

# Panel B — FID over epochs
axes[1].plot(fid_rows["Epoch"], fid_rows["FID"],
             color=col("HYBRID"), lw=2.2, marker="o", markersize=6)
axes[1].axhline(fid_rows["FID"].min(), color=col("HYBRID"),
                lw=1.2, ls=":", alpha=0.7,
                label=f"Best FID = {fid_rows['FID'].min():.2f}")
axes[1].set_title("B  FID score", fontweight="bold")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("FID")
axes[1].legend(fontsize=9)

# Panel C — Mode Variance
axes[2].plot(ep, mv_s, color=col("HYBRID"), lw=2.0)
axes[2].set_title("C  Mode Variance", fontweight="bold")
axes[2].set_xlabel("Epoch")
axes[2].set_ylabel("Mode Variance")

fig.suptitle("Hybrid Loss on CheXpert  (100 epochs, real data)",
             fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()
save_fig(fig, "10_chexpert_hybrid_analysis")


  saved → plots/10_chexpert_hybrid_analysis.png


## Run All Plots at Once

In [29]:
# Re-run this cell to regenerate ALL 10 plots in one go
import importlib, runpy, os

plots_to_run = [
    ("01_fid_curves_cifar10",    lambda: exec(open("plots_cells/plot01.py").read()) if False else None),
]

# ── Inline runner ─────────────────────────────────────────────────────────────
def run_all():
    from scipy.ndimage import uniform_filter1d as ufi

    def sm(a, w=5): return ufi(np.array(a,dtype=float), size=min(w,len(a)))

    all_figs = {}

    # 01
    fig, ax = plt.subplots(figsize=(11,5.5))
    for k in ORDER:
        ep,fid = get_fid(CIFAR,k)
        ax.plot(ep,fid,color=col(k),lw=2.5 if k=="HYBRID" else 1.8,
                ls="--" if k=="HYBRID" else "-",marker="o",
                markersize=6 if k=="HYBRID" else 4,label=lbl(k))
    ax.set(xlabel="Epoch",ylabel="FID (↓)",
           title="FID Curves — CIFAR-10")
    ax.legend(fontsize=9,ncol=2)
    plt.tight_layout(); save_fig(fig,"01_fid_curves_cifar10"); plt.close()

    # 02
    fig, ax = plt.subplots(figsize=(11,5.5))
    for k in ["STANDARD","LSGAN","WGAN","WGANGP","HINGE"]:
        ep,fid = get_fid(EURO150,k)
        ax.plot(ep,fid,color=col(k),lw=1.8,marker="o",markersize=4,label=lbl(k))
    eh = EURO_HYB[EURO_HYB["FID"].notna()]
    ax.plot(eh["Epoch"],eh["FID"],color=col("HYBRID"),lw=2.5,ls="--",
            marker="o",markersize=6,label=lbl("HYBRID"))
    ax.set(xlabel="Epoch",ylabel="FID (↓)",title="FID Curves — EuroSAT")
    ax.legend(fontsize=9,ncol=2)
    plt.tight_layout(); save_fig(fig,"02_fid_curves_eurosat"); plt.close()

    # 03
    fig,axes = plt.subplots(2,3,figsize=(15,8)); axes=axes.flatten()
    for ax,k in zip(axes,ORDER):
        ep=CIFAR[CIFAR["Experiment"]==k]["Epoch"].tolist()
        ax.plot(ep,sm(get_col(CIFAR,k,"G_Loss")),color=col(k),lw=2,label="Generator")
        ax.plot(ep,sm(get_col(CIFAR,k,"D_Loss")),color=col(k),lw=1.4,ls="--",alpha=0.6,label="Discriminator")
        ax.set_title(lbl(k),color=col(k),fontweight="bold")
        ax.set_xlabel("Epoch",fontsize=9); ax.set_ylabel("Loss",fontsize=9)
        ax.legend(fontsize=8)
    fig.suptitle("Loss Curves — CIFAR-10",fontsize=13,fontweight="bold",y=1.01)
    plt.tight_layout(); save_fig(fig,"03_loss_curves_cifar10"); plt.close()

    # 04
    dfs={k:EURO150[EURO150["Experiment"]==k] for k in ["STANDARD","LSGAN","WGAN","WGANGP","HINGE"]}
    dfs["HYBRID"]=EURO_HYB
    fig,axes=plt.subplots(2,3,figsize=(15,8)); axes=axes.flatten()
    for ax,k in zip(axes,ORDER):
        s=dfs[k]; ep=s["Epoch"].tolist()
        ax.plot(ep,sm(s["G_Loss"].tolist()),color=col(k),lw=2,label="Generator")
        ax.plot(ep,sm(s["D_Loss"].tolist()),color=col(k),lw=1.4,ls="--",alpha=0.6,label="Discriminator")
        ax.set_title(lbl(k),color=col(k),fontweight="bold")
        ax.set_xlabel("Epoch",fontsize=9); ax.set_ylabel("Loss",fontsize=9)
        ax.legend(fontsize=8)
    fig.suptitle("Loss Curves — EuroSAT",fontsize=13,fontweight="bold",y=1.01)
    plt.tight_layout(); save_fig(fig,"04_loss_curves_eurosat"); plt.close()

    # 05
    fig,ax=plt.subplots(figsize=(11,5.5))
    for k in ORDER:
        ep=CIFAR[CIFAR["Experiment"]==k]["Epoch"].tolist()
        ax.plot(ep,sm(get_col(CIFAR,k,"ModeVar"),w=7),color=col(k),
                lw=2.5 if k=="HYBRID" else 1.8,ls="--" if k=="HYBRID" else "-",label=lbl(k))
    ax.set(xlabel="Epoch",ylabel="Mode Variance (↑)",title="Mode Variance — CIFAR-10")
    ax.legend(fontsize=9,ncol=2)
    plt.tight_layout(); save_fig(fig,"05_modevar_curves_cifar10"); plt.close()

    # 06
    dfs={k:EURO150[EURO150["Experiment"]==k] for k in ["STANDARD","LSGAN","WGAN","WGANGP","HINGE"]}
    dfs["HYBRID"]=EURO_HYB
    fig,ax=plt.subplots(figsize=(11,5.5))
    for k in ORDER:
        s=dfs[k]; ep=s["Epoch"].tolist()
        ax.plot(ep,sm(s["ModeVar"].tolist(),w=7),color=col(k),
                lw=2.5 if k=="HYBRID" else 1.8,ls="--" if k=="HYBRID" else "-",label=lbl(k))
    ax.set(xlabel="Epoch",ylabel="Mode Variance (↑)",title="Mode Variance — EuroSAT")
    ax.legend(fontsize=9,ncol=2)
    plt.tight_layout(); save_fig(fig,"06_modevar_curves_eurosat"); plt.close()

    # 07
    bc={k:min(get_fid(CIFAR,k)[1]) for k in ORDER}
    be={k:min(get_fid(EURO150,k)[1]) for k in ["STANDARD","LSGAN","WGAN","WGANGP","HINGE"]}
    be["HYBRID"]=float(EURO_HYB[EURO_HYB["FID"].notna()]["FID"].min())
    bx={k:None for k in ORDER}; bx["HYBRID"]=float(CHEX_HYB[CHEX_HYB["FID"].notna()]["FID"].min())
    x=np.arange(len(ORDER)); w=0.26
    fig,ax=plt.subplots(figsize=(13,6))
    for bars,vals,a,h,lbl_ in [
        (ax.bar(x-w,[bc[k] for k in ORDER],w,color=[col(k) for k in ORDER],edgecolor="white",zorder=3),None,1.0,"","CIFAR-10"),
        (ax.bar(x,  [be[k] for k in ORDER],w,color=[col(k) for k in ORDER],edgecolor="white",alpha=0.6,hatch="///",zorder=3),None,0.6,"///","EuroSAT"),
        (ax.bar(x+w,[bx[k] if bx[k] else 0 for k in ORDER],w,color=[col(k) for k in ORDER],edgecolor="white",alpha=0.35,hatch="xxx",zorder=3),None,0.35,"xxx","CheXpert"),
    ]:
        for b in bars:
            v=b.get_height()
            if v>1: ax.text(b.get_x()+b.get_width()/2,v+1.5,f"{v:.1f}",ha="center",va="bottom",fontsize=7.5,color="#444")
    ax.set_xticks(x); ax.set_xticklabels([LABELS[k] for k in ORDER],rotation=18,ha="right")
    ax.set_ylabel("Best FID (↓)"); ax.set_title("Best FID — All Datasets")
    ax.legend(handles=[Line2D([0],[0],color="#666",lw=8,alpha=a,label=l) for a,l in [(1.0,"CIFAR-10"),(0.6,"EuroSAT"),(0.35,"CheXpert")]],fontsize=9)
    plt.tight_layout(); save_fig(fig,"07_best_fid_all_datasets"); plt.close()

    # 08
    mc={k:CIFAR[CIFAR["Experiment"]==k]["ModeVar"].iloc[-1] for k in ORDER}
    me={k:EURO150[EURO150["Experiment"]==k]["ModeVar"].iloc[-1] for k in ["STANDARD","LSGAN","WGAN","WGANGP","HINGE"]}
    me["HYBRID"]=EURO_HYB["ModeVar"].iloc[-1]
    x=np.arange(len(ORDER)); w=0.35
    fig,ax=plt.subplots(figsize=(12,5.5))
    for b in [ax.bar(x-w/2,[mc[k] for k in ORDER],w,color=[col(k) for k in ORDER],edgecolor="white",zorder=3),
              ax.bar(x+w/2,[me[k] for k in ORDER],w,color=[col(k) for k in ORDER],alpha=0.55,edgecolor="white",hatch="///",zorder=3)]:
        for bar in b: ax.text(bar.get_x()+bar.get_width()/2,bar.get_height()+0.003,f"{bar.get_height():.3f}",ha="center",va="bottom",fontsize=7.5,color="#444")
    ax.set_xticks(x); ax.set_xticklabels([LABELS[k] for k in ORDER],rotation=18,ha="right")
    ax.set_ylabel("Mode Variance (↑)"); ax.set_title("Final Mode Variance — CIFAR-10 vs EuroSAT")
    ax.legend(handles=[Line2D([0],[0],color="#666",lw=8,alpha=a,label=l) for a,l in [(1.0,"CIFAR-10"),(0.55,"EuroSAT")]],fontsize=9)
    plt.tight_layout(); save_fig(fig,"08_modevar_bar_comparison"); plt.close()

    # 09
    bf={k:min(get_fid(CIFAR,k)[1]) for k in ORDER}
    fm={k:CIFAR[CIFAR["Experiment"]==k]["ModeVar"].iloc[-1] for k in ORDER}
    fig,ax=plt.subplots(figsize=(9,6))
    for k in ORDER:
        ax.scatter(bf[k],fm[k],color=col(k),s=280 if k=="HYBRID" else 120,
                   marker="*" if k=="HYBRID" else "o",edgecolors="white",linewidths=0.8,zorder=5)
        ax.annotate(LABELS[k],(bf[k],fm[k]),textcoords="offset points",
                    xytext=(5,-14 if k=="STANDARD" else 5),fontsize=9,color=col(k),fontweight="bold")
    ax.set(xlabel="Best FID (←)",ylabel="Mode Variance (↑)",
           title="Quality–Diversity Scatter — CIFAR-10  ★=Hybrid")
    plt.tight_layout(); save_fig(fig,"09_pareto_scatter_cifar10"); plt.close()

    # 10
    ep=CHEX_HYB["Epoch"].tolist()
    fid_r=CHEX_HYB[CHEX_HYB["FID"].notna()]
    fig,axes=plt.subplots(1,3,figsize=(15,5))
    axes[0].plot(ep,sm(CHEX_HYB["G_Loss"].tolist(),w=9),color=col("HYBRID"),lw=2,label="Generator")
    axes[0].plot(ep,sm(CHEX_HYB["D_Loss"].tolist(),w=9),color=col("HYBRID"),lw=1.5,ls="--",alpha=0.6,label="Discriminator")
    axes[0].set_title("A  Loss curves",fontweight="bold"); axes[0].set_xlabel("Epoch"); axes[0].legend(fontsize=9)
    axes[1].plot(fid_r["Epoch"],fid_r["FID"],color=col("HYBRID"),lw=2.2,marker="o",markersize=6)
    axes[1].axhline(fid_r["FID"].min(),color=col("HYBRID"),lw=1.2,ls=":",alpha=0.7,label=f"Best={fid_r['FID'].min():.2f}")
    axes[1].set_title("B  FID score",fontweight="bold"); axes[1].set_xlabel("Epoch"); axes[1].legend(fontsize=9)
    axes[2].plot(ep,sm(CHEX_HYB["ModeVar"].tolist(),w=7),color=col("HYBRID"),lw=2)
    axes[2].set_title("C  Mode Variance",fontweight="bold"); axes[2].set_xlabel("Epoch")
    fig.suptitle("Hybrid Loss — CheXpert  (100 epochs, real data)",fontsize=13,fontweight="bold",y=1.01)
    plt.tight_layout(); save_fig(fig,"10_chexpert_hybrid_analysis"); plt.close()

    print("\nAll 10 plots regenerated successfully.")

run_all()


  saved → plots/01_fid_curves_cifar10.png
  saved → plots/02_fid_curves_eurosat.png
  saved → plots/03_loss_curves_cifar10.png
  saved → plots/04_loss_curves_eurosat.png
  saved → plots/05_modevar_curves_cifar10.png
  saved → plots/06_modevar_curves_eurosat.png
  saved → plots/07_best_fid_all_datasets.png
  saved → plots/08_modevar_bar_comparison.png
  saved → plots/09_pareto_scatter_cifar10.png
  saved → plots/10_chexpert_hybrid_analysis.png

All 10 plots regenerated successfully.


In [30]:
# cd /d/mtech/sem 4/gen Ai/gitcode/Comparative\ Analysis\ of\ GAN\ Loss\ Functions ; python --version ; python - <<'PY'
import os
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

csv_path = os.path.join('.', 'csv', 'cifer_all_6.csv')
df = pd.read_csv(csv_path)
df['FID'] = df['FID'].fillna(method='ffill')

best_fid = df.dropna(subset=['FID']).groupby('Experiment')['FID'].min().reset_index().sort_values('FID')

plt.figure(figsize=(10, 6))
bars = plt.bar(best_fid['Experiment'], best_fid['FID'], color=['#4C78A8', '#F58518', '#54A24B', '#EECA3B', '#B279A2', '#FF9DA7'])
for bar in bars:
    h = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, h + 2, f'{h:.2f}', ha='center', va='bottom', fontsize=10)
plt.title('Best FID Comparison Across Loss Functions', fontsize=14, weight='bold')
plt.xlabel('Loss Function', fontsize=12)
plt.ylabel('Best FID', fontsize=12)
plt.grid(axis='y', linestyle='--', alpha=0.4)
plt.tight_layout()
out_path = os.path.join('.', 'plots', 'best_fid_comparison.png')
os.makedirs(os.path.dirname(out_path), exist_ok=True)
plt.savefig(out_path, dpi=300, bbox_inches='tight')
plt.close()
print(out_path)
print(best_fid.to_string(index=False))


.\plots\best_fid_comparison.png
Experiment      FID
     LSGAN  42.2016
      WGAN  42.2016
     HINGE  50.2412
    WGANGP  50.2412
  STANDARD  56.8606
    HYBRID 110.9571
